In [1]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import re
from datasets import load_dataset, DatasetDict 

from vllm import LLM, SamplingParams

from tqdm import tqdm

from llm_fairness.utils import load_resume_data, load_resume_data_with_race, load_llm_and_tokenizer, gen_prompt, DictToObject

import argparse
import warnings

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from trl import (
    DatasetMixtureConfig,
    DPOConfig,
    DPOTrainer,
    ModelConfig,
    ScriptArguments,
    TrlParser,
    get_dataset,
    get_kbit_device_map,
    get_peft_config,
    get_quantization_config,
)
from trl.trainer.utils import SIMPLE_CHAT_TEMPLATE

INFO 08-21 16:29:44 [__init__.py:244] Automatically detected platform cuda.


In [ ]:
# pip install "transformers<4.54.0"

In [2]:
def make_parser(subparsers: argparse._SubParsersAction = None):
    dataclass_types = (ScriptArguments, DPOConfig, ModelConfig, DatasetMixtureConfig)
    if subparsers is not None:
        parser = subparsers.add_parser("dpo", help="Run the DPO training script", dataclass_types=dataclass_types)
    else:
        parser = TrlParser(dataclass_types)
    return parser


In [ ]:
parser = make_parser()

script_args, training_args, model_args, dataset_args, _ = parser.parse_args_and_config(
        return_remaining_strings=True
    )

In [6]:
import pandas as pd
import numpy as np

# --- Inputs you already have ---
scores_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_scores.csv")
scores_df = scores_df.squeeze()
responses_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_summaries.csv")
responses_df = responses_df.drop(columns=['responses'])

# 1) Parse scores → keep only the value numbers (skip enumerations 1., 2., ...)
scores_series = scores_df.squeeze()
numbers_expanded = (
    scores_series
        .str.findall(r'\d+')          # all numbers as strings
        .apply(lambda lst: lst[1::2]) # keep value numbers only
        .apply(lambda x: list(map(int, x)))  # to int
)

# 2) Expand to a scores DataFrame aligned to response columns
numbers_df = pd.DataFrame(numbers_expanded.tolist(), index=scores_series.index)

# Ensure same column order/names as responses_df
numbers_df.columns = responses_df.columns

# 3) Build a mask of valid (non-empty) response cells
#    Treat NaN or pure-whitespace/empty strings as invalid
resp_stripped = responses_df.applymap(lambda v: np.nan if pd.isna(v) else str(v).strip())
valid_mask = resp_stripped.notna() & (resp_stripped != "")

# 4) Mask out scores where the response cell is empty for that row
masked_scores = numbers_df.where(valid_mask, other=np.nan)

# 5) Max/min column names per row (skipna=True ignores masked NaNs)
max_cols = masked_scores.idxmax(axis=1, skipna=True)
min_cols = masked_scores.idxmin(axis=1, skipna=True)

# 6) Convert responses_df to NumPy for fast row/col selection
data = responses_df.to_numpy()
cols = list(responses_df.columns)

def col_to_pos(col_name):
    return cols.index(col_name) if isinstance(col_name, str) else -1

max_pos = max_cols.map(col_to_pos).to_numpy()
min_pos = min_cols.map(col_to_pos).to_numpy()

rows = np.arange(len(responses_df))

# 7) Gather chosen/rejected values; put None if a row had no valid responses
max_values = [data[i, max_pos[i]] if max_pos[i] >= 0 else None for i in rows]
min_values = [data[i, min_pos[i]] if min_pos[i] >= 0 else None for i in rows]

dpo_training_pairs = pd.DataFrame({
    'max_value': max_values,
    'min_value': min_values
}, index=responses_df.index)

print(dpo_training_pairs.iloc[0])

# Save
dpo_training_pairs.to_csv('/local/zemel/arvind/code/llm_fairness/data/dpo_training_pairs.csv', index=False)


max_value    [NAME] is a seasoned public safety professiona...
min_value    [NAME], a dedicated and experienced public ser...
Name: 0, dtype: object


/tmp/ipykernel_2813531/1456943983.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  resp_stripped = responses_df.applymap(lambda v: np.nan if pd.isna(v) else str(v).strip())
